# Практика 5 Детекция и мультиобъектный трекинг

## 0 Среда и воспроизводимая мини сцена

Нужны только `numpy`, `scipy`, `pandas` и `matplotlib`.

Формат бокса в ноутбуке: `(x1, y1, x2, y2)`. У детекции есть пятая колонка `score`. Координаты нормированы не будут: это пиксели условного кадра 100×70.

In [ ]:
%matplotlib inline
from dataclasses import dataclass, field
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment

RNG = np.random.default_rng(17)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.2})


def make_scene(n_frames=50, seed=17):
    # Возвращает два словаря frame -> ndarray.
    # GT: [object_id, x1, y1, x2, y2].
    # detections: [x1, y1, x2, y2, score].
    rng = np.random.default_rng(seed)
    gt, detections = {}, {}
    for t in range(n_frames):
        boxes = np.array([
            [1,  6 + 1.35*t, 18 + 0.10*t, 18 + 1.35*t, 38 + 0.10*t],
            [2, 82 - 1.25*t, 20 - 0.04*t, 94 - 1.25*t, 40 - 0.04*t],
            [3, 42 + 0.18*t, 48 - 0.55*t, 54 + 0.18*t, 66 - 0.55*t],
        ], dtype=float)
        gt[t] = boxes
        frame_dets = []
        for object_id, x1, y1, x2, y2 in boxes:
            # При пересечении объектов 1 и 2 слабее виден объект 1.
            low = object_id == 1 and 24 <= t <= 31
            score = rng.uniform(0.18, 0.48) if low else rng.uniform(0.78, 0.98)
            # Два полных пропуска делают задачу реалистичнее.
            if (object_id, t) in {(1, 27), (3, 37)}:
                continue
            noise = rng.normal(0, 0.9 if not low else 1.4, size=4)
            frame_dets.append([x1+noise[0], y1+noise[1], x2+noise[2], y2+noise[3], score])
        # Редкие ложные детекции с невысоким score.
        if t in {12, 29, 43}:
            x, y = rng.uniform(5, 80), rng.uniform(5, 50)
            frame_dets.append([x, y, x+9, y+15, rng.uniform(0.20, 0.42)])
        detections[t] = np.asarray(frame_dets, dtype=float).reshape(-1, 5)
    return gt, detections


GT, DETECTIONS = make_scene()


def draw_frame(frame, gt=None, detections=None, tracks=None, ax=None, title=None):
    ax = ax or plt.gca()
    ax.set_xlim(0, 100); ax.set_ylim(70, 0); ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    if gt is not None:
        for object_id, x1, y1, x2, y2 in gt:
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                   edgecolor="black", linestyle="--", linewidth=1.5))
            ax.text(x1, y1-1, f"GT {int(object_id)}", color="black", fontsize=8)
    if detections is not None:
        for x1, y1, x2, y2, score in detections:
            color = "tab:green" if score >= 0.6 else "tab:orange"
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                   edgecolor=color, linewidth=2))
            ax.text(x1, y2+2, f"{score:.2f}", color=color, fontsize=8)
    if tracks is not None:
        for x1, y1, x2, y2, track_id in tracks:
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                   edgecolor="tab:blue", linewidth=2.5))
            ax.text(x1, y1-2, f"ID {int(track_id)}", color="tab:blue", weight="bold")
    ax.set_title(title or f"Кадр {frame}")
    return ax

### Входной эксперимент

На кадрах 22, 27 и 31 происходит пересечение. Зелёные боксы имеют высокий score, оранжевые — низкий. Пунктиром показана истина, которая обычно недоступна трекеру.

Какой из подходов скорее сохранит ID объекта 1?

1. удалить все детекции со `score < 0.6`;
2. использовать слабые детекции для продолжения существующего трека, но не рождать из них новые треки;
3. рождать новый трек из каждой слабой детекции.

Запишите вариант и одно предложение с обоснованием. К ответу вернёмся после реализации ByteTrack.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, t in zip(axes, [22, 27, 31]):
    draw_frame(t, GT[t], DETECTIONS[t], ax=ax)
plt.tight_layout()

## 1 IoU и глобальное назначение

Для боксов $A$ и $B$:

$$IoU(A,B)=\frac{|A\cap B|}{|A\cup B|}.$$

В трекере удобно минимизировать стоимость $C_{ij}=1-IoU(\hat b_i,d_j)$, где $\hat b_i$ — прогноз трека, $d_j$ — детекция. `linear_sum_assignment` минимизирует **сумму** стоимостей по всем выбранным парам. Это важнее, чем локально взять лучшую детекцию для каждого трека.

### Задание 1 Векторная IoU матрица

Реализуйте `iou_matrix(a, b)` без циклов по парам. Учтите пустые входы и зажимайте ширину и высоту пересечения снизу нулём.

<details><summary>Подсказка</summary>
Добавьте осям форму `a[:, None, :]` и `b[None, :, :]`. Левый верхний угол пересечения — покоординатный максимум, правый нижний — минимум.
</details>

In [ ]:
def iou_matrix(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=float)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    intersection = wh[..., 0] * wh[..., 1]
    area_a = np.clip(a[:, 2]-a[:, 0], 0, None) * np.clip(a[:, 3]-a[:, 1], 0, None)
    area_b = np.clip(b[:, 2]-b[:, 0], 0, None) * np.clip(b[:, 3]-b[:, 1], 0, None)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)

In [ ]:
a = np.array([[0, 0, 10, 10], [20, 20, 30, 30]])
b = np.array([[0, 0, 10, 10], [5, 0, 15, 10]])
got = iou_matrix(a, b)
expected = np.array([[1.0, 1/3], [0.0, 0.0]])
print(np.round(got, 3))
assert got.shape == (2, 2)
assert np.allclose(got, expected)
assert iou_matrix(np.empty((0, 4)), b).shape == (0, 2)

### Задание 2 Венгерское назначение с порогом

Реализуйте `associate`. Сначала получите глобально оптимальные пары, затем отвергните пары с `IoU < min_iou`. Верните список пар `(track_index, detection_index)` и индексы несопоставленных элементов.

Почему порог нужен после назначения? Венгерский алгоритм всегда старается назначить пары, даже если все они плохие. В реальном трекере плохая пара означает «трек временно потерян» и «детекция, возможно, новый объект».

<details><summary>Подсказка</summary>
После `rows, cols = linear_sum_assignment(1 - ious)` оставьте только пары `ious[r, c] >= min_iou`, затем соберите дополнения множеств индексов.
</details>

In [ ]:
def associate(track_boxes, det_boxes, min_iou=0.2):
    track_boxes = np.asarray(track_boxes, dtype=float).reshape(-1, 4)
    det_boxes = np.asarray(det_boxes, dtype=float).reshape(-1, 4)
    if len(track_boxes) == 0 or len(det_boxes) == 0:
        return [], list(range(len(track_boxes))), list(range(len(det_boxes)))
    ious = iou_matrix(track_boxes, det_boxes)
    rows, cols = linear_sum_assignment(1 - ious)
    matches = [(int(r), int(c)) for r, c in zip(rows, cols) if ious[r, c] >= min_iou]
    used_r = {r for r, _ in matches}; used_c = {c for _, c in matches}
    unmatched_tracks = [i for i in range(len(track_boxes)) if i not in used_r]
    unmatched_dets = [j for j in range(len(det_boxes)) if j not in used_c]
    return matches, unmatched_tracks, unmatched_dets

In [ ]:
# Локальный выбор T1->D1 (0.10) оставляет T2->D2 (0.90), сумма 1.00.
# Глобальный выбор T1->D2 (0.20), T2->D1 (0.11) даёт 0.31.
cost_demo = np.array([[0.10, 0.20], [0.11, 0.90]])
r, c = linear_sum_assignment(cost_demo)
print("Глобальное назначение:", list(zip(r, c)), "сумма =", cost_demo[r, c].sum())
assert list(zip(r, c)) == [(0, 1), (1, 0)]

matches, ut, ud = associate(a, b, min_iou=0.25)
print("matches:", matches, "unmatched tracks:", ut, "unmatched detections:", ud)
assert matches == [(0, 0)] and ut == [1] and ud == [1]

## 2 Фильтр Калмана в окклюзии

Начнём с 1D-состояния $x=[p,v]^T$: положение и скорость. При $\Delta t=1$:

$$F=\begin{bmatrix}1&1\\0&1\end{bmatrix},\qquad H=\begin{bmatrix}1&0\end{bmatrix}.$$

**Predict** переносит состояние вперёд и увеличивает неопределённость:

$$x^- = Fx,\qquad P^- = FPF^T+Q.$$

**Update** сравнивает измерение $z$ с прогнозом. Инновация $y=z-Hx^-$, её ковариация $S=HP^-H^T+R$, усиление Калмана $K=P^-H^TS^{-1}$:

$$x=x^-+Ky,\qquad P=(I-KH)P^-.$$

### Задание 3 Реализовать predict и update

Во время кадров 18–25 измерения отсутствуют. В эти кадры вызывается только `predict`. Перед запуском ответьте: что произойдёт с шириной коридора $\pm2\sigma$ и почему?

<details><summary>Подсказка</summary>
В `predict` обе строки буквально повторяют формулы с `F` и `Q`. В `update` следите за формами: `K` имеет форму 2×1, а `y` — один элемент.
</details>

In [ ]:
class Kalman1D:
    def __init__(self, position, q=0.10, r=4.0):
        self.x = np.array([position, 0.0], dtype=float)
        self.P = np.diag([25.0, 16.0])
        self.F = np.array([[1.0, 1.0], [0.0, 1.0]])
        self.H = np.array([[1.0, 0.0]])
        self.Q = q * np.array([[0.25, 0.5], [0.5, 1.0]])
        self.R = np.array([[r]])

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return float(self.x[0])

    def update(self, measurement):
        z = np.array([float(measurement)])
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + (K @ y).ravel()
        self.P = (np.eye(2) - K @ self.H) @ self.P
        return float(self.x[0])

In [ ]:
rng = np.random.default_rng(5)
t_axis = np.arange(45)
truth = 8 + 1.25*t_axis
measurements = truth + rng.normal(0, 2.0, size=len(t_axis))
measurements[18:26] = np.nan

kf = Kalman1D(measurements[0])
estimates, sigmas = [], []
for z in measurements:
    estimate = kf.predict()
    if np.isfinite(z):
        estimate = kf.update(z)
    estimates.append(estimate)
    sigmas.append(np.sqrt(kf.P[0, 0]))

estimates, sigmas = np.array(estimates), np.array(sigmas)
plt.figure(figsize=(11, 4))
plt.plot(t_axis, truth, "k--", label="истина")
plt.scatter(t_axis, measurements, s=22, color="tab:orange", label="измерения")
plt.plot(t_axis, estimates, color="tab:blue", label="оценка")
plt.fill_between(t_axis, estimates-2*sigmas, estimates+2*sigmas,
                 color="tab:blue", alpha=0.18, label="±2σ")
plt.axvspan(17.5, 25.5, color="gray", alpha=0.12, label="окклюзия")
plt.xlabel("кадр"); plt.ylabel("координата"); plt.legend(ncol=5); plt.show()

print(f"σ до окклюзии: {sigmas[17]:.2f}; максимум в окклюзии: {sigmas[25]:.2f}; после update: {sigmas[26]:.2f}")
assert sigmas[25] > sigmas[17]
assert sigmas[26] < sigmas[25]

## 3 Минимальный SORT

Полный SORT хранит Калман-фильтр для каждого бокса. Класс `Track`: он оценивает скорость центра по двум последним наблюдениям и сдвигает бокс в `predict`.

- `age` — сколько кадров живёт трек;
- `hits` — сколько раз он получил детекцию;
- `missed` — сколько кадров подряд нет сопоставленной детекции;
- новый трек подтверждается после `min_hits` наблюдений;
- трек удаляется, когда `missed > max_age`.

### Задание 4 Реализовать один шаг трекера

Заполните `OneStageTracker.update` по шести шагам: прогноз, фильтрация детекций, назначение, обновление совпавших, рождение новых, удаление старых. Не меняйте порядок: он является архитектурой tracking-by-detection.

<details><summary>Подсказка</summary>
`predicted = np.asarray([tr.predict() for tr in self.tracks]).reshape(-1, 4)`. Несопоставленные старые треки специально не обновляются: их `missed` уже увеличен в `predict`.
</details>

In [ ]:
@dataclass
class Track:
    box: np.ndarray
    track_id: int
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2))
    age: int = 1
    hits: int = 1
    missed: int = 0
    last_center: np.ndarray | None = None

    def predict(self):
        shift = np.tile(self.velocity, 2)
        self.box = self.box + shift
        self.age += 1
        self.missed += 1
        return self.box.copy()

    def update(self, box):
        box = np.asarray(box, dtype=float)
        center = (box[:2] + box[2:]) / 2
        previous = (self.box[:2] + self.box[2:]) / 2
        measured_velocity = center - previous
        self.velocity = 0.65*self.velocity + 0.35*measured_velocity
        self.last_center = previous
        self.box = box.copy()
        self.hits += 1
        self.missed = 0


class OneStageTracker:
    def __init__(self, score_thresh=0.6, min_iou=0.15, max_age=2, min_hits=2):
        self.score_thresh = score_thresh
        self.min_iou = min_iou
        self.max_age = max_age
        self.min_hits = min_hits
        self.tracks = []
        self.next_id = 1

    def _new_track(self, box):
        self.tracks.append(Track(np.asarray(box, dtype=float).copy(), self.next_id))
        self.next_id += 1

    def visible(self):
        rows = [np.r_[tr.box, tr.track_id] for tr in self.tracks
                if tr.missed == 0 and tr.hits >= self.min_hits]
        return np.asarray(rows, dtype=float).reshape(-1, 5)


    def update(self, detections):
        detections = np.asarray(detections, dtype=float).reshape(-1, 5)
        predicted = np.asarray([tr.predict() for tr in self.tracks], dtype=float).reshape(-1, 4)
        selected = detections[detections[:, 4] >= self.score_thresh]
        matches, _, unmatched_dets = associate(predicted, selected[:, :4], self.min_iou)
        for track_idx, det_idx in matches:
            self.tracks[track_idx].update(selected[det_idx, :4])
        for det_idx in unmatched_dets:
            self._new_track(selected[det_idx, :4])
        self.tracks = [tr for tr in self.tracks if tr.missed <= self.max_age]
        return self.visible()

In [ ]:
def run_tracker(tracker, detections):
    outputs = {}
    for frame in sorted(detections):
        outputs[frame] = tracker.update(detections[frame])
    return outputs


def show_track_sequence(outputs, frames=(22, 27, 31, 34)):
    fig, axes = plt.subplots(1, len(frames), figsize=(4.2*len(frames), 4))
    for ax, t in zip(np.atleast_1d(axes), frames):
        draw_frame(t, GT[t], tracks=outputs[t], ax=ax)
    plt.tight_layout()


sort_outputs = run_tracker(OneStageTracker(score_thresh=0.6, max_age=2), DETECTIONS)
show_track_sequence(sort_outputs)

**Наблюдение.** Найдите ID объекта 1 до и после окклюзии. Если ID изменился, разделите причину на два решения:

1. какой порог удалил полезные детекции;
2. какое правило жизненного цикла успело удалить трек.

**Мини эксперимент** Запустите one-stage трекер с `max_age=10`. Стало ли этого достаточно? Объясните, почему «дольше хранить всё» и «использовать слабые наблюдения» — не одно и то же.

In [ ]:
sort_age10_outputs = run_tracker(OneStageTracker(score_thresh=0.6, max_age=10), DETECTIONS)
show_track_sequence(sort_age10_outputs)

## 4 Второй раунд ByteTrack

ByteTrack различает три действия:

1. сопоставить активные треки с $D_{high}$ (`score ≥ 0.6`);
2. оставшиеся треки сопоставить с $D_{low}$ (`0.1 ≤ score < 0.6`);
3. новые треки рождать **только** из несопоставленных $D_{high}$.

### Задание 5 Реализовать второй раунд

Заполните `ByteLiteTracker.update`. Во втором раунде участвуют только треки, не сопоставленные в первом. Внимательно переводите локальный индекс остатка обратно в индекс `self.tracks`.

<details><summary>Подсказка</summary>
Сначала получите `unmatched_tracks` из первого раунда. Второй `associate` вернёт локальный индекс в массиве `predicted[unmatched_tracks]`; исходный индекс равен `unmatched_tracks[local_index]`.
</details>

In [ ]:
class ByteLiteTracker(OneStageTracker):
    def __init__(self, high_thresh=0.6, low_thresh=0.1, **kwargs):
        super().__init__(score_thresh=high_thresh, **kwargs)
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh


    def update(self, detections):
        detections = np.asarray(detections, dtype=float).reshape(-1, 5)
        predicted = np.asarray([tr.predict() for tr in self.tracks], dtype=float).reshape(-1, 4)
        high = detections[detections[:, 4] >= self.high_thresh]
        low = detections[(detections[:, 4] >= self.low_thresh) &
                         (detections[:, 4] < self.high_thresh)]

        first, unmatched_tracks, unmatched_high = associate(predicted, high[:, :4], self.min_iou)
        for track_idx, det_idx in first:
            self.tracks[track_idx].update(high[det_idx, :4])

        remaining_boxes = predicted[unmatched_tracks] if unmatched_tracks else np.empty((0, 4))
        second_local, _, _ = associate(remaining_boxes, low[:, :4], self.min_iou)
        for local_track_idx, det_idx in second_local:
            track_idx = unmatched_tracks[local_track_idx]
            self.tracks[track_idx].update(low[det_idx, :4])

        for det_idx in unmatched_high:
            self._new_track(high[det_idx, :4])
        self.tracks = [tr for tr in self.tracks if tr.missed <= self.max_age]
        return self.visible()

In [ ]:
byte_outputs = run_tracker(ByteLiteTracker(max_age=2, min_iou=0.15), DETECTIONS)
show_track_sequence(byte_outputs)

## 5 Метрики качество детекции и постоянство

Сначала на каждом кадре сопоставим GT и результат при `IoU ≥ 0.5`.

- `TP`, `FP`, `FN` описывают качество обнаружения.
- `IDSW` растёт, если один GT-объект после сопоставления получает другой предсказанный ID.
- $MOTA=1-(FN+FP+IDSW)/GT$ смешивает три типа ошибок и может быть отрицательной.
- `IDF1` строит одно глобальное соответствие GT-ID ↔ predicted-ID на всём видео; поэтому строго наказывает разорванную личность.
- `HOTA` разделяет детекцию (`DetA`) и ассоциацию (`AssA`), а затем объединяет их геометрическим средним.

In [ ]:
def evaluate_tracks(gt, outputs, min_iou=0.5):
    tp = fp = fn = idsw = 0
    total_gt = total_pred = 0
    previous_pred_for_gt = {}
    pair_counts = defaultdict(int)
    fragments = defaultdict(int)
    was_matched = defaultdict(bool)

    for frame in sorted(gt):
        gt_rows = gt[frame]
        pred_rows = outputs.get(frame, np.empty((0, 5)))
        total_gt += len(gt_rows); total_pred += len(pred_rows)
        matches, unmatched_gt, unmatched_pred = associate(gt_rows[:, 1:5], pred_rows[:, :4], min_iou)
        tp += len(matches); fn += len(unmatched_gt); fp += len(unmatched_pred)

        matched_gt_ids = set()
        for gi, pi in matches:
            gid, pid = int(gt_rows[gi, 0]), int(pred_rows[pi, 4])
            pair_counts[(gid, pid)] += 1
            matched_gt_ids.add(gid)
            if gid in previous_pred_for_gt and previous_pred_for_gt[gid] != pid:
                idsw += 1
            if not was_matched[gid] and gid in previous_pred_for_gt:
                fragments[gid] += 1
            previous_pred_for_gt[gid] = pid
            was_matched[gid] = True
        for row in gt_rows:
            gid = int(row[0])
            if gid not in matched_gt_ids:
                was_matched[gid] = False

    gt_ids = sorted({gid for gid, _ in pair_counts})
    pred_ids = sorted({pid for _, pid in pair_counts})
    overlap = np.zeros((len(gt_ids), len(pred_ids)), dtype=int)
    for (gid, pid), count in pair_counts.items():
        overlap[gt_ids.index(gid), pred_ids.index(pid)] = count
    if overlap.size:
        rows, cols = linear_sum_assignment(-overlap)
        idtp = int(overlap[rows, cols].sum())
    else:
        idtp = 0
    idfn, idfp = total_gt-idtp, total_pred-idtp
    idf1 = 2*idtp / max(2*idtp + idfp + idfn, 1)
    mota = 1 - (fn + fp + idsw) / max(total_gt, 1)
    return {
        "MOTA": mota, "IDF1": idf1, "IDSW": idsw,
        "FP": fp, "FN": fn, "fragments": sum(fragments.values()),
        "precision": tp/max(tp+fp, 1), "recall": tp/max(tp+fn, 1),
    }


comparison = pd.DataFrame({
    "SORT high only": evaluate_tracks(GT, sort_outputs),
    "SORT max_age=10": evaluate_tracks(GT, sort_age10_outputs),
    "ByteLite 2 rounds": evaluate_tracks(GT, byte_outputs),
}).T
display(comparison.round({"MOTA": 3, "IDF1": 3, "precision": 3, "recall": 3}))

1. Какая настройка сильнее всего уменьшила `FN`?
2. Почему увеличение `max_age` может сохранить ID, но не восстановить пропущенные выходные боксы?
3. Почему у ByteLite могут улучшиться `IDF1` и `IDSW`, хотя детектор тот же?
4. Придумайте ситуацию, где второй раунд увеличит число ошибочных ассоциаций. Какой дополнительный сигнал мог бы помочь?

1. Почему жадный выбор ближайшей детекции может проиграть венгерскому алгоритму?
2. Что происходит с ковариацией Калмана без измерений?
3. Почему ByteTrack не рождает новые треки из `D_low`?
4. Какая метрика лучше покажет, что объект сохранил один ID: MOTA или IDF1? Почему?